## 🧬 Cloning Tables en Databricks

En Data Engineering es común encontrarnos con escenarios donde necesitamos trabajar sobre una copia de una tabla sin afectar la información original.

Para ello, Databricks incorpora la funcionalidad **Clone**, que permite crear una nueva Delta Table a partir de otra existente.

Sin embargo, antes de clonar una tabla debemos responder una pregunta:

> 🤔 ¿Necesitamos únicamente una nueva definición lógica de la tabla o una copia completamente independiente de sus datos?

La respuesta determinará el tipo de clonación que debemos utilizar. A continuación, se muestran los 2 tiposde clonación:

* ⚡ Shallow Clone: Crea una nueva Delta Table reutilizando los archivos físicos de la tabla origen. En otras palabras:

    * ✅ Se crea una nueva definición de la tabla.
    * ✅ Se generan nuevos metadatos para la tabla clonada.
    * ❌ No se copian los archivos Parquet.

    La nueva tabla simplemente referencia los mismos archivos físicos que utiliza la tabla original.

    * Sintaxis:
        ```sql
        CREATE TABLE catalog.schema.tabla_clonada
        SHALLOW CLONE catalog.schema.tabla_origen;
        ```

    * 📌 ¿Cuándo utilizarlo?:

        * Ambientes temporales de desarrollo.
        * Pruebas rápidas.
        * Análisis sin duplicar almacenamiento.
        * Crear una copia de forma prácticamente instantánea.

        Al no copiar los datos físicos, el proceso es muy rápido y consume muy poco espacio de almacenamiento.

---

* 🏗️ Deep Clone: Crea una copia completamente independiente de la tabla original. Durante el proceso, Databricks:

    * ✅ Copia los metadatos.
    * ✅ Copia todos los archivos físicos.
    * ✅ Genera una nueva Delta Table totalmente autónoma.

    * Sintaxis

        ```sql
        CREATE TABLE catalog.schema.tabla_clonada
        DEEP CLONE catalog.schema.tabla_origen;
        ```

    Opcionalmente podemos indicar dónde almacenar físicamente la copia:

    ```sql
    CREATE TABLE catalog.schema.tabla_clonada
    DEEP CLONE catalog.schema.tabla_origen
    LOCATION 'ruta_de_almacenamiento';
    ```

---

### 📍 El papel de `LOCATION`

Aquí encontramos una diferencia muy interesante. Aunque la documentación muestra que `LOCATION` puede formar parte de la sintaxis general de creación de tablas, en la práctica:

* **Shallow Clone** no utiliza una ubicación de almacenamiento propia, debido que reutiliza los archivos de la tabla origen y, genera una Managed Table.

* **Deep Clone** sí puede definir una ubicación física distinta mediante `LOCATION`, debido que durante la clonación se generan nuevos archivos de datos.

Esto permite decidir quién administrará físicamente la información clonada.
Por un lado:
* Sin `LOCATION`: Databricks administra automáticamente los archivos físicos de la tabla clonada.
* Con `LOCATION`: Los datos se almacenan en la ubicación especificada, permitiendo un mayor control sobre el almacenamiento físico.

---

### 🤯 Un detalle muy interesante

Cuando trabajamos con una **Managed Table**, normalmente no visualizamos directamente la estructura física donde Databricks almacena los archivos de la tabla.

Sin embargo, al realizar un **Deep Clone** especificando una `LOCATION`, la nueva tabla queda asociada a una ruta de almacenamiento conocida.

Esto nos permite inspeccionar directamente su contenido y observar elementos como:

* 📦 Archivos Parquet
* 📝 Directorio `_delta_log`
* 📜 Archivos JSON del historial transaccional

En otras palabras, además de obtener una copia completamente independiente, también podemos explorar con mayor facilidad la estructura interna de una Delta Table.

---

> 💡 La diferencia entre ambos enfoques no radica únicamente en la velocidad de clonación, sino en el nivel de independencia que tendrá la nueva tabla respecto a la original. Mientras **Shallow Clone** reutiliza los datos existentes para ahorrar tiempo y almacenamiento, **Deep Clone** genera una copia completa, ideal para respaldos, migraciones o entornos completamente aislados.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("10CloningTables").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### 🎩 Shallow Clone

In [0]:
### PASO 1). VERIFICAMOS TABLA ORIGEN
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk"))

### PASO 2). APLICAR SHALLOW CLONE
spark.sql("""
          
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.categories_shallow
          SHALLOW CLONE catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk
          
          """)
print("Shallow Clone aplicado correctamente")

### PASO 3. VERIFICAR TABLA CLONADA
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.categories_shallow"))

### 🧬 Deep Clone

In [0]:
### PASO 1). VERIFICAMOS TABLA ORIGEN
# display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk"))

"""

    💡 Si queremos que Databricks gestiones el almacenamiento de los archivos de la delta table clonada.

"""

### PASO 1.5  APLICAR DEEP CLONE (DATABRICKS GESTIONA ALMACENAMIENTO)
# spark.sql("""
          
#           CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.categories_deep
#           DEEP CLONE catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk
          
#           """)
# print("Deep Clone aplicado correctamente")    

### PASO 2. VERIFICAR TABLA CLONADA
# display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.categories_deep"))

###================================================================================================

"""

    💡 Si queremos que el almacenamiento de los archivos de la delta table clonada
        sea gestionada de manera externa (S3).

"""

### PASO 1.5). REALIZAR CONFIGURACIÓN DE EXTERNAL LOCATION Y STORAGE CREDENTIAL

#### Revisar: "https://github.com/BrayanR03/Databricks-DE-2026/blob/main/assets/pasos-previos-external-tables.md"

### PASO 2). APLICAR DEEP CLONE
spark.sql("""
          
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.categories_deep
          DEEP CLONE catalog_databricks_2026_de.schema_databricks_2026_de.categories_fk
          LOCATION 's3://bucket-datasets-brayan/cloning_tables/deep_clone_table/'

          """)
print("Deep clone ejecutado exitosamente")

### PASO 3. VERIFICAR TABLA CLONADA
display(spark.sql("SELECT * FROM catalog_databricks_2026_de.schema_databricks_2026_de.categories_deep"))